In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scenarios_and_methods import (
    generate_two_gaussian_abrupt_shift, generate_two_gaussian_slow_shift,
    generate_exchangeable_gaussian, generate_var1_dependence, generate_complex_dependence,
    risk_monitoring, plugin_martingale_test, simple_jumper_martingale_test,
    continuous_pairwise_betting_martingale_test,
    v_test, v_test_2d, v_test_distance, durbin_watson_test, yule_walker_test,
    binomial_martingale_test, aggressive_martingale_test, binomial_mixture_martingale_test, permutation_pvalue
)

# Perform tests

In [ ]:
scenarios_dict = {
    "exchangeable": generate_exchangeable_gaussian,
    "two_gaussian_abrupt_shift": generate_two_gaussian_abrupt_shift,
    "generate_two_gaussian_slow_shift": generate_two_gaussian_slow_shift,
    "generate_var1_dependence": generate_var1_dependence,
    "generate_complex_dependence": generate_complex_dependence,
}

methods_dict = {
    "plugin_martingale": plugin_martingale_test,
    "jumper_martingale": simple_jumper_martingale_test,
    "continuous_pairwise_betting_martingale": continuous_pairwise_betting_martingale_test,
    "risk_monitoring": risk_monitoring,
    "v_test": v_test,
    "v_test_2d": v_test_2d,
    "v_test_distance": v_test_distance,
    "durbin_watson_test": durbin_watson_test,
    "yule_walker_test": yule_walker_test,
    "permutation_pvalue": permutation_pvalue,
    "binomial_martingale_test": binomial_martingale_test,
    "aggressive_martingale_test": aggressive_martingale_test,
    "binomial_mixture_martingale_test": binomial_mixture_martingale_test,
}

methods_type = {
    "plugin_martingale": "sequential",
    "jumper_martingale": "sequential",
    "continuous_pairwise_betting_martingale": "sequential",
    "risk_monitoring": "sequential",
    "v_test": "batch",
    "v_test_2d": "batch",
    "v_test_distance": "batch",
    "durbin_watson_test": "batch",
    "yule_walker_test": "batch",
    "permutation_pvalue": "batch",
    "binomial_martingale_test": "batch",
    "aggressive_martingale_test": "batch",
    "binomial_mixture_martingale_test": "batch",
}

scenario_pretty_names = {
    "exchangeable": "Exchangeable Gaussian",
    "two_gaussian_abrupt_shift": "Two Gaussian Abrupt Shift",
    "generate_two_gaussian_slow_shift": "Two Gaussian Slow Shift",
    "generate_var1_dependence": "VAR(1) Dependence",
    "generate_complex_dependence": "Complex Dependence",
}

method_pretty_names = {
    "plugin_martingale": "Plugin Martingale",
    "jumper_martingale": "Jumper Martingale",
    "continuous_pairwise_betting_martingale": "Continuous Pairwise Betting",
    "risk_monitoring": "Risk Monitoring",
    "v_test": "V-Test",
    "v_test_2d": "V-Test 2D",
    "v_test_distance": "V-Test Distance",
    "durbin_watson_test": "Durbin-Watson",
    "yule_walker_test": "Yule-Walker",
    "permutation_pvalue": "Permutation P-Value",
    "binomial_martingale_test": "Binomial Martingale",
    "aggressive_martingale_test": "Aggressive Martingale",
    "binomial_mixture_martingale_test": "Binomial Mixture",
}

n_samples = 500
n_runs = 30

result_list = []
for scenario_name, generate_data in scenarios_dict.items():
    for run in range(n_runs):
        is_exchangeable_ground_truth, X_to_test, y_to_test, X_train, y_train = generate_data(
            n_samples=n_samples
        )

        for method_name, method in methods_dict.items():
            is_exchangeable, stable_index, stable_value = method(
                X_to_test=X_to_test,
                y_to_test=y_to_test,
                X_train=X_train,
                y_train=y_train,
            )
            result_list.append(
                {
                    "scenario": scenario_name,
                    "method": method_name,
                    "run": run,
                    "is_exchangeable_ground_truth": int(is_exchangeable_ground_truth),
                    "is_exchangeable": int(is_exchangeable),
                    "stable_index": stable_index,
                    "stable_value": stable_value,
                }
            )

df = pd.DataFrame(result_list)


# Show results

In [ ]:
# Aggregate one value per run
run_level_df = (
    df.groupby(["scenario", "method", "run"])
    .agg(
        is_exchangeable_ground_truth=("is_exchangeable_ground_truth", "first"),
        is_exchangeable=("is_exchangeable", "first"),
    )
    .reset_index()
)

# Mean/std over runs per scenario/method
summary_df = (
    run_level_df.groupby(["scenario", "method"])
    .agg(
        is_exchangeable_ground_truth=("is_exchangeable_ground_truth", "first"),
        is_exchangeable_mean=("is_exchangeable", "mean"),
        is_exchangeable_std=("is_exchangeable", "std"),
    )
    .reset_index()
)
summary_df["is_exchangeable_std"] = summary_df["is_exchangeable_std"].fillna(0.0)

# Format table entries as "mean +- std"
summary_df["mean_std"] = summary_df.apply(
    lambda row: f"{row['is_exchangeable_mean']:.3f} (std {row['is_exchangeable_std']:.3f})",
    axis=1,
)

# Build display table with methods in rows, scenarios in columns
scenario_order = list(scenarios_dict.keys())
method_order = list(methods_dict.keys())

scenario_pretty_order = [scenario_pretty_names.get(s, s) for s in scenario_order]
method_pretty_order = [method_pretty_names.get(m, m) for m in method_order]

table_df = summary_df.pivot(index="method", columns="scenario", values="mean_std")
table_df = table_df.reindex(index=method_order, columns=scenario_order)
table_df.index = [method_pretty_names.get(m, m) for m in table_df.index]
table_df.columns = [scenario_pretty_names.get(s, s) for s in table_df.columns]

gt_df = summary_df[["scenario", "is_exchangeable_ground_truth"]].drop_duplicates()
gt_row = (
    gt_df.set_index("scenario")
    .reindex(scenario_order)["is_exchangeable_ground_truth"]
    .astype(int)
    .astype(str)
)

final_table_df = pd.concat(
    [
        pd.DataFrame([gt_row.values], index=["Ground Truth"], columns=scenario_pretty_order),
        table_df,
    ]
)


In [ ]:
# Create a small grid plot summarizing correctness per method/scenario
# A method predicts exchangeability with majority vote over runs:
# is_exchangeable_pred = mean(is_exchangeable) > 0.5
pred_df = summary_df.copy()
pred_df["is_exchangeable_pred"] = pred_df["is_exchangeable_mean"] > 0.5

scenario_order = list(scenarios_dict.keys())
method_order = list(methods_dict.keys())
scenario_pretty_order = [scenario_pretty_names.get(s, s) for s in scenario_order]
method_pretty_order = [method_pretty_names.get(m, m) for m in method_order]

gt_series = (
    gt_df.set_index("scenario")
    .reindex(scenario_order)["is_exchangeable_ground_truth"]
    .astype(bool)
)

# Numeric correctness grid + text values shown in each cell
grid = []
cell_text = []
for method in method_order:
    row = []
    row_text = []
    for scenario in scenario_order:
        sel = pred_df[
            (pred_df["method"] == method) & (pred_df["scenario"] == scenario)
        ]["is_exchangeable_pred"]
        pred_value = bool(sel.iloc[0]) if len(sel) else False
        gt_value = bool(gt_series.loc[scenario])
        row.append(1 if pred_value == gt_value else 0)

        val_sel = summary_df[
            (summary_df["method"] == method) & (summary_df["scenario"] == scenario)
        ]["mean_std"]
        row_text.append(val_sel.iloc[0] if len(val_sel) else "")
    grid.append(row)
    cell_text.append(row_text)
grid = np.array(grid)

fig, ax = plt.subplots(
    figsize=(2 + 1.2 * len(scenario_order), 1 + 0.55 * len(method_order))
)

# show as colored squares: green for correct, black for incorrect
from matplotlib.colors import ListedColormap
cmap = ListedColormap(["black", "green"])
ax.imshow(grid, cmap=cmap, aspect="auto")

# Overlay values inside each cell
for i in range(len(method_order)):
    for j in range(len(scenario_order)):
        ax.text(
            j,
            i,
            cell_text[i][j],
            ha="center",
            va="center",
            fontsize=8,
            color="white",
        )

# Set ticks and labels: scenarios in columns, methods in rows
ax.set_xticks(np.arange(len(scenario_order)))
ax.set_yticks(np.arange(len(method_order)))
ax.set_xticklabels(scenario_pretty_order, rotation=45, ha="right")
ax.set_yticklabels(method_pretty_order)
ax.set_xlabel("Scenario")
ax.set_ylabel("Method")
ax.set_title("Correct Exchangeability Detection (Green = correct, Black = incorrect)")

# Minor grid for clarity
ax.set_xticks(np.arange(-0.5, len(scenario_order), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(method_order), 1), minor=True)
ax.grid(which="minor", color="gray", linestyle="-", linewidth=0.5, alpha=0.15)
ax.tick_params(which="minor", length=0)

plt.tight_layout()
plt.show()


# Plot stats over time

In [ ]:
# One figure per method: rows are data / stable_value / stable_index, columns are scenarios.
scenario_names = list(scenarios_dict.keys())
method_names = list(methods_dict.keys())

for method_name in method_names:
    n_rows = 3
    n_cols = len(scenario_names)
    fig, axes = plt.subplots(
        nrows=n_rows,
        ncols=n_cols,
        figsize=(4.5 * n_cols, 9),
        squeeze=False,
    )

    for j, scenario_name in enumerate(scenario_names):
        scenario_func = scenarios_dict[scenario_name]
        res = scenario_func()

        # Row 1: scenario data
        ax_data = axes[0, j]
        if len(res) == 5:
            _, X_to_test, y_to_test, _, _ = res
            if X_to_test.ndim == 2 and X_to_test.shape[1] >= 2:
                scatter = ax_data.scatter(
                    X_to_test[:, 0],
                    X_to_test[:, 1],
                    c=y_to_test,
                    cmap='coolwarm',
                    alpha=0.6,
                )
                if j == 0:
                    legend1 = ax_data.legend(*scatter.legend_elements(), title='Class')
                    ax_data.add_artist(legend1)
                ax_data.set_xlabel('X[:, 0]')
                ax_data.set_ylabel('X[:, 1]')
            else:
                xvals = X_to_test[:, 0] if X_to_test.ndim == 2 else X_to_test
                ax_data.scatter(
                    xvals,
                    np.zeros_like(xvals),
                    c=y_to_test,
                    cmap='coolwarm',
                    alpha=0.6,
                )
                ax_data.set_xlabel('X[:, 0]')
                ax_data.set_ylabel('')
        else:
            ax_data.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax_data.transAxes)
            ax_data.set_xticks([])
        ax_data.set_title(scenario_pretty_names.get(scenario_name, scenario_name))
        if j == 0:
            ax_data.set_ylabel('Data')

        df_cell = df[(df['method'] == method_name) & (df['scenario'] == scenario_name)]

        # Row 2: violin of stable_value
        ax_value = axes[1, j]
        stable_value_vals = df_cell['stable_value'].dropna().astype(float).values
        if stable_value_vals.size > 0:
            vp_val = ax_value.violinplot([stable_value_vals], showmeans=True, showextrema=True)
            for body in vp_val['bodies']:
                body.set_alpha(0.5)
            ax_value.set_xticks([1])
            ax_value.set_xticklabels(['Stable Value'], rotation=20)
        else:
            ax_value.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax_value.transAxes)
            ax_value.set_xticks([])
        if j == 0:
            ax_value.set_ylabel('Stable Value')

        # Row 3: violin of stable_index
        ax_index = axes[2, j]
        stable_index_vals = df_cell['stable_index'].dropna().astype(float).values
        if stable_index_vals.size > 0:
            vp_idx = ax_index.violinplot([stable_index_vals], showmeans=True, showextrema=True)
            for body in vp_idx['bodies']:
                body.set_alpha(0.5)
            ax_index.set_xticks([1])
            ax_index.set_xticklabels(['Stable Index'], rotation=20)
        else:
            ax_index.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax_index.transAxes)
            ax_index.set_xticks([])
        if j == 0:
            ax_index.set_ylabel('Stable Index')

    plt.suptitle(f"Method: {method_pretty_names.get(method_name, method_name)}", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
# Compare stable_index across selected fixed-dataset permutation methods
methods_to_compare = [
    "permutation_pvalue",
    "binomial_martingale_test",
    "aggressive_martingale_test",
    "binomial_mixture_martingale_test",
]

df_compare = df[df["method"].isin(methods_to_compare)].copy()

# Summary table
stable_index_summary = (
    df_compare.groupby(["scenario", "method"], as_index=False)
    .agg(
        stable_index_mean=("stable_index", "mean"),
        stable_index_std=("stable_index", "std"),
        n_runs=("stable_index", "count"),
    )
    .sort_values(["scenario", "method"])
)
stable_index_summary["stable_index_std"] = stable_index_summary["stable_index_std"].fillna(0.0)
stable_index_summary["scenario"] = stable_index_summary["scenario"].map(
    lambda s: scenario_pretty_names.get(s, s)
)
stable_index_summary["method"] = stable_index_summary["method"].map(
    lambda m: method_pretty_names.get(m, m)
)
stable_index_summary

# Violin comparison by scenario
n_scenarios = len(scenarios_dict)
fig, axes = plt.subplots(1, n_scenarios, figsize=(5 * n_scenarios, 4), squeeze=False)

for j, scenario_name in enumerate(scenarios_dict.keys()):
    ax = axes[0, j]
    data = []
    labels = []

    for method_name in methods_to_compare:
        vals = df_compare[
            (df_compare["scenario"] == scenario_name)
            & (df_compare["method"] == method_name)
        ]["stable_index"].dropna().astype(float).values
        if vals.size > 0:
            data.append(vals)
            labels.append(method_pretty_names.get(method_name, method_name))

    if len(data) > 0:
        vp = ax.violinplot(data, showmeans=True, showextrema=True)
        for body in vp["bodies"]:
            body.set_alpha(0.5)
        ax.set_xticks(np.arange(1, len(labels) + 1))
        ax.set_xticklabels(labels, rotation=30, ha="right")
    else:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_xticks([])

    ax.set_title(scenario_pretty_names.get(scenario_name, scenario_name))
    ax.set_ylabel("Stable Index")

plt.suptitle("Stable index comparison for fixed-dataset methods", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Compare stable_index across online/sequential methods
methods_to_compare_online = [
    "plugin_martingale",
    "jumper_martingale",
    "continuous_pairwise_betting_martingale",
    "risk_monitoring",
]

df_compare_online = df[df["method"].isin(methods_to_compare_online)].copy()

# Summary table
stable_index_summary_online = (
    df_compare_online.groupby(["scenario", "method"], as_index=False)
    .agg(
        stable_index_mean=("stable_index", "mean"),
        stable_index_std=("stable_index", "std"),
        n_runs=("stable_index", "count"),
    )
    .sort_values(["scenario", "method"])
)
stable_index_summary_online["stable_index_std"] = stable_index_summary_online[
    "stable_index_std"
].fillna(0.0)
stable_index_summary_online["scenario"] = stable_index_summary_online["scenario"].map(
    lambda s: scenario_pretty_names.get(s, s)
)
stable_index_summary_online["method"] = stable_index_summary_online["method"].map(
    lambda m: method_pretty_names.get(m, m)
)
stable_index_summary_online

# Violin comparison by scenario
n_scenarios = len(scenarios_dict)
fig, axes = plt.subplots(1, n_scenarios, figsize=(5 * n_scenarios, 4), squeeze=False)

for j, scenario_name in enumerate(scenarios_dict.keys()):
    ax = axes[0, j]
    data = []
    labels = []

    for method_name in methods_to_compare_online:
        vals = df_compare_online[
            (df_compare_online["scenario"] == scenario_name)
            & (df_compare_online["method"] == method_name)
        ]["stable_index"].dropna().astype(float).values
        if vals.size > 0:
            data.append(vals)
            labels.append(method_pretty_names.get(method_name, method_name))

    if len(data) > 0:
        vp = ax.violinplot(data, showmeans=True, showextrema=True)
        for body in vp["bodies"]:
            body.set_alpha(0.5)
        ax.set_xticks(np.arange(1, len(labels) + 1))
        ax.set_xticklabels(labels, rotation=30, ha="right")
    else:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_xticks([])

    ax.set_title(scenario_pretty_names.get(scenario_name, scenario_name))
    ax.set_ylabel("Stable Index")

plt.suptitle("Stable index comparison for online/sequential methods", fontsize=14)
plt.tight_layout()
plt.show()
